<a href="https://colab.research.google.com/github/apester/TDA/blob/main/TDA_Lecture_4_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TDA Lecture 4 — Exercises

This notebook contains exercises related to:
- topology as input features
- topology-aware losses
- latent-space analysis
- NLP embeddings

Try to solve the tasks before expanding the hints or writing your own code.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.decomposition import PCA
from sklearn.datasets import make_circles, make_moons, make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors

In [2]:
def pairwise_distances(X):
    X = np.asarray(X, dtype=float)
    diff = X[:, None, :] - X[None, :, :]
    return np.sqrt(np.sum(diff**2, axis=-1))

def knn_graph(X, k=6):
    X = np.asarray(X, dtype=float)
    nbrs = NearestNeighbors(n_neighbors=min(k+1, len(X))).fit(X)
    distances, indices = nbrs.kneighbors(X)
    edges = set()
    for i in range(len(X)):
        for j in indices[i][1:]:
            a, b = sorted((i, int(j)))
            edges.add((a, b))
    return sorted(edges)

def connected_components(n, edges):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra
    for a, b in edges:
        union(a, b)
    groups = {}
    for i in range(n):
        r = find(i)
        groups.setdefault(r, []).append(i)
    return list(groups.values())

def count_triangles(n, edges):
    edge_set = set(tuple(sorted(e)) for e in edges)
    c = 0
    for i, j, k in combinations(range(n), 3):
        if ((i, j) in edge_set and (i, k) in edge_set and (j, k) in edge_set):
            c += 1
    return c

def cycle_rank(n, edges):
    comps = len(connected_components(n, edges))
    return len(edges) - n + comps

def crude_topology_features(X, k=6):
    edges = knn_graph(X, k=k)
    n = len(X)
    comps = len(connected_components(n, edges))
    triangles = count_triangles(n, edges)
    c_rank = cycle_rank(n, edges)
    avg_deg = 2 * len(edges) / n
    return np.array([n, len(edges), comps, c_rank, triangles, avg_deg], dtype=float)

## Exercise 1 — Compare shapes

Generate three datasets:
- circles
- moons
- blobs

Task:
1. plot them
2. compute `crude_topology_features`
3. compare the `cycle_rank` and `components`

Which dataset looks most “loop-like”?

In [ ]:
# Your solution here

## Exercise 2 — Build a simple topology-based classifier

Create repeated samples of:
- circles
- moons
- blobs

Use the crude topological features as input to a classifier.

Tasks:
1. define a binary label (for example circle vs not-circle)
2. train a model
3. report accuracy
4. interpret which feature seems most informative

In [ ]:
# Your solution here

## Exercise 3 — Pixel loss vs topology penalty

Construct:
- a target binary mask
- an imperfect prediction with either a broken object or an extra disconnected region

Tasks:
1. compute mean squared pixel loss
2. compute number of connected components in target and prediction
3. define a toy topology penalty
4. combine both into a total loss

What does the topology term capture that the pixel term misses?

In [ ]:
# Your solution here

## Exercise 4 — Latent representation quality

Make a structured dataset, map it into a higher-dimensional ambient space, then compress it to 2D with PCA.

Tasks:
1. compare the original and latent scatter plots
2. compute crude topological features in both spaces
3. discuss whether the latent code preserved important structure

In [ ]:
# Your solution here

## Exercise 5 — Higher-order interactions

Build a kNN graph from a point cloud.

Tasks:
1. count edges
2. count triangles
3. explain why triangles represent higher-order structure rather than only pairwise relations
4. name two applications where this matters

In [ ]:
# Your solution here

## Exercise 6 — NLP embedding geometry

Create a toy embedding space for a few words from different semantic groups.

Tasks:
1. reduce to 2D
2. plot the words
3. inspect whether clusters appear
4. explain how TDA could be used as a structural diagnostic for this space

In [ ]:
# Your solution here

## Extension questions

1. Why are differentiable topological losses difficult to optimize?
2. In which domains is topology-aware segmentation especially valuable?
3. Why might topology be more useful for *representation analysis* than for *raw prediction* in NLP?
4. How could you compare two language models using topological summaries of their embeddings?